# 04 · anatomy: registration and the Allen atlas

### CAJAL NEUROMICS summer school, Bordeaux 2026 · a spatial metabolomics primer

*Luca Fusar Bassini · level 2, notebook 4 · ~90 minutes*

---

So far every pixel has been a little spectrum floating in space. We know its lipids, we
know which section it came from, we even know where it sits on the slide. What we do not
yet know is the one thing a neuroscientist asks first: **where in the brain is it?** Is
this pixel in the caudoputamen or the hippocampus, in grey matter or in a white-matter
tract, in cortical layer 1 or layer 6? Without that label, a lipid map is a pretty
picture. With it, the map becomes anatomy, and anatomy is what lets us compare control to
pregnant region by region, and later join our lipids to MERFISH gene expression in the
exact same regions.

This notebook is about how every pixel gets an anatomical address. The trick is called
**registration to a reference atlas**, and the reference is the **Allen Mouse Brain Common
Coordinate Framework, version 3 (CCFv3)**. We will:

1. understand what a reference atlas is, and why we need per-pixel anatomical coordinates,
2. understand registration as an **affine transform plus a nonlinear warp**, built from a tiny
   bit of linear algebra you can hold in your head,
3. meet **ABBA**, the community-standard tool for doing this, as a concept (we ship its
   output, we do not run it here),
4. load the per-pixel CCF coordinates that already live on our two sections, and see how a
   coordinate **indexes** the Allen annotation volume to give each pixel a region name and color,
5. draw region maps for control vs pregnant, split grey from white matter, and
6. build **region-level lipid views**, the mean lipid profile of every Allen region, which is
   the table that powers the region-level differential test and the MERFISH gene join later.

The whole notebook runs on the two sections you already know: **217D**, a control female,
and **Brain1_C2**, a pregnant female, cut at the same coronal plane near AP 6.5.


## the callouts

The same four markers run through every notebook:

- 🔬 **TASK**: something you do (write or run code).
- 💡 **HINT**: a nudge when you are stuck.
- ❓ **QUESTION**: pause and think; no code required.
- ⚠️ **CHECKPOINT**: what you should see if it worked. If your screen disagrees, stop and fix it.

🔬 **TASK.** Run the next cell to load the libraries and our helper package. You met the
scientific-Python stack in the intro notebooks; `cajal_lipidomics` is the small course
package whose plotting and analysis functions we lean on, after we have unrolled the idea
behind each one by hand.


In [ ]:
# the stack you know, plus our course helper package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import ast  # to parse the Allen structure path, which is stored as a string

import cajal_lipidomics as cl
from cajal_lipidomics import plotting, analysis
from cajal_lipidomics.style import set_style, FS
set_style()  # the course-wide clean figure style

# one global seed so every number and figure below is reproducible
RNG_SEED = 0
np.random.seed(RNG_SEED)

print("ready. cajal_lipidomics", cl.__version__)


⚠️ **CHECKPOINT.** You should see `ready. cajal_lipidomics 0.0.1` and no red error. If you
see a `ModuleNotFoundError`, your notebook is on the wrong kernel; pick the
`cajal-lipidomics` kernel from the top-right kernel picker.


## 1 · what is a reference atlas, and why do we need one

A pixel is one MALDI laser spot, about 25 micrometers across. The mass spectrometer hands
us, for each pixel, an intensity for every lipid. It does **not** hand us a brain region.
The scanner only knows that this spot was at row `x`, column `y` of the slide. That
slide-pixel coordinate is meaningless across sections: the tissue was placed at a slightly
different angle, the section was cut at a slightly different depth, the brain was a little
bigger or smaller. Two pixels with the same `(x, y)` on two slides are almost never the same
piece of brain.

We need a **shared coordinate system** that every section can be mapped into, so that "this
location" means the same thing in control and in pregnant, in your brain and in mine. That
shared system is a **reference atlas**.

The **Allen Mouse Brain Common Coordinate Framework v3 (CCFv3)** is the standard one. Think
of it as two things bolted together:

- **a 3D coordinate space.** The whole mouse brain is described as a box of cubic voxels, each
  **25 micrometers** on a side. A point in the brain is just three numbers, `(x, y, z)`, in
  millimeters, measuring how far along the three anatomical axes you are. The axes are the
  **anterior-posterior (AP)** axis (front to back), the **dorsal-ventral** axis (top to
  bottom), and the **medial-lateral** axis (midline to side). This is the same idea as latitude,
  longitude, and altitude on Earth: any place gets one fixed triple of numbers.
- **an annotation volume.** For every one of those voxels, the Allen team has filled in **which
  brain region it belongs to**. The brain was carved, by experts, into a tree of about a
  thousand structures: big divisions like cortex, striatum, thalamus, then finer ones like
  caudoputamen, then finer still like cortical layers. Each region has a short **acronym**
  (`CP` for caudoputamen), a full **name** ("Caudoputamen"), and a **color** chosen so that
  related regions look alike.

So once a pixel has a CCF coordinate `(x, y, z)`, getting its region is a pure **lookup**: go
to that voxel in the annotation volume, read off the region. Coordinate first, anatomy second.
The hard part, and the subject of this notebook, is getting that coordinate. That is
**registration**: bending each real, distorted section until it lines up with the atlas, then
reading each pixel's atlas coordinate.

❓ **QUESTION.** Why can we not just measure region by eye on each section? You can, for big
obvious structures. But our analysis compares hundreds of fine regions across two sections
and later against a third dataset (MERFISH). Doing that consistently, pixel by pixel, by eye,
across thousands of pixels, is hopeless. A shared coordinate frame does it once and exactly.


## 2 · registration as an affine transform plus a nonlinear warp

Registration is the act of finding the geometric transformation that takes your section onto
the atlas. It comes in two layers, and the layers matter, so we build the intuition from the
linear algebra up.

### the affine layer: rotate, scale, shear, translate

The first layer handles the *gross* misalignment: your section is rotated a few degrees,
slightly bigger or smaller than the atlas, maybe squashed along one axis because the tissue
shrank, and shifted off-center. Every one of those is an **affine transformation**: a linear
map (a 2x2 matrix `A` in 2D) followed by a translation (a vector `t`). A point `p` moves to

$$ p' = A\,p + t. $$

The matrix `A` packs four operations into four numbers: rotation, isotropic scaling
(same in both directions), anisotropic scaling (different per axis, which is how you undo a
20% shrinkage along one direction), and shear. The translation `t` slides the whole thing.
Affine maps keep straight lines straight and parallel lines parallel: a grid drawn on the
section stays a grid, just rotated, stretched, and sheared. That is the key limitation, and
why we need a second layer.

🔬 **TASK.** Build the intuition with a tiny demo. We take a small square grid of points,
apply one affine transform (a rotation, an anisotropic scale, and a shift), and look at what
happened. No atlas yet, just the geometry.


In [ ]:
# a small grid of points: this stands in for "landmarks on a section"
gx, gy = np.meshgrid(np.linspace(0, 1, 6), np.linspace(0, 1, 6))
pts = np.column_stack([gx.ravel(), gy.ravel()])  # (36 points, 2 coords)

# build one affine transform A p + t
theta = np.deg2rad(20)                            # rotate 20 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.diag([1.3, 0.7])                            # stretch x by 1.3, squash y to 0.7
A = R @ S                                          # compose: first scale, then rotate
t = np.array([0.4, -0.2])                          # then translate

pts_affine = pts @ A.T + t                         # apply to every point at once

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts[:, 0], pts[:, 1], s=18, c="0.6", label="original grid (section)")
ax.scatter(pts_affine[:, 0], pts_affine[:, 1], s=18, c="crimson",
           label="after affine A·p + t")
ax.set_aspect("equal"); ax.legend(fontsize=FS["s"])
ax.set_title("affine transform: rotate, scale, shear, translate")
plt.tight_layout(); plt.show()
print("A =\n", np.round(A, 2), "\nt =", t)


⚠️ **CHECKPOINT.** The red grid is a rotated, stretched, shifted copy of the gray grid, but
it is **still a perfect grid**: straight lines stayed straight, parallel stayed parallel.
That is exactly what an affine map can and cannot do.

❓ **QUESTION.** Real brain tissue does not just rotate and stretch uniformly. A section gets
torn, folded, locally swollen near a ventricle, compressed where the blade dragged. Can any
single matrix `A` undo a local fold in one corner while leaving the rest untouched? No, one
matrix acts on the whole plane identically. That is the job of the second layer.


### the nonlinear layer: a smooth warp that bends locally

After the affine has done the gross alignment, regions are roughly in place but the fine
boundaries still do not match: the hippocampus is a touch too curved here, the cortex a bit
too thick there. The second layer is a **nonlinear warp**, a smooth deformation field that can
push each point a different little amount, bending locally without tearing.

Formally it is a **displacement field**: at every location it stores a small arrow saying
"move this bit of tissue by this much, in this direction". The good algorithms constrain the
field to be **smooth and invertible** (a *diffeomorphism*), so tissue never folds over itself
or rips. Methods like elastix splines (used by ABBA) or LDDMM (used by STalign) learn this
field by minimizing the mismatch between the warped section and the atlas, while penalizing
roughness.

🔬 **TASK.** Add a gentle nonlinear warp on top of the affine grid, so you can see the
difference. We push each point by a smooth sine-shaped displacement: now the lines bend.


In [ ]:
# take the affine-transformed grid and add a smooth, local, nonlinear displacement
x, y = pts_affine[:, 0], pts_affine[:, 1]
amp = 0.12  # how strong the warp is
dx = amp * np.sin(2.0 * np.pi * y)   # horizontal push depends smoothly on y
dy = amp * np.sin(2.0 * np.pi * x)   # vertical push depends smoothly on x
pts_warped = np.column_stack([x + dx, y + dy])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts_affine[:, 0], pts_affine[:, 1], s=18, c="crimson",
           label="affine only (lines stay straight)")
ax.scatter(pts_warped[:, 0], pts_warped[:, 1], s=22, c="navy",
           label="affine + nonlinear warp (lines bend)")
# draw the displacement arrows to make the warp visible
ax.quiver(pts_affine[:, 0], pts_affine[:, 1], dx, dy,
          angles="xy", scale_units="xy", scale=1, width=0.004, color="0.5")
ax.set_aspect("equal"); ax.legend(fontsize=FS["s"])
ax.set_title("nonlinear warp: a smooth displacement field on top of the affine")
plt.tight_layout(); plt.show()


⚠️ **CHECKPOINT.** The blue points no longer form straight rows: the gray arrows show each
point being nudged a little, differently in different places, but smoothly. That bending is
what lets the section's curved hippocampus snap onto the atlas's curved hippocampus.

So the full registration is: **affine first** to get the gross fit, **nonlinear warp second**
to bend the fine boundaries into place. Run it, and you can read off, for every pixel, the
atlas coordinate it landed on. That coordinate is the CCF triple `(xccf, yccf, zccf)` we will
load next.


## 3 · ABBA: the community-standard tool (a concept, not a button here)

In a real lab you do not write the warp by hand. You use a tool. The community standard for
registering serial brain sections to the Allen CCFv3 is **ABBA**, which stands for **Aligning
Big Brains and Atlases**, from Nicolas Chiaruttini's group at the EPFL bioimaging platform
(Chiaruttini et al., *Cell Reports* 2025, doi:10.1016/j.celrep.2025.115876).

ABBA is a **Fiji (ImageJ) plugin**, usually driven together with **QuPath**, with a
mouse-driven graphical interface. You feed it your section images, place them roughly along
the atlas, and it runs exactly the two layers we just built:

- an optional machine-learning pre-alignment (**DeepSlice**) guesses which atlas slice you are
  looking at and the cutting angle,
- an **affine elastix** step does the gross rotate/scale/shear/translate fit,
- a **spline elastix** step does the nonlinear warp,
- and for stubborn slices you hand-place landmark pairs in a tool called **BigWarp**.

A crucial detail for our data: ABBA also corrects the **slicing angle**. A coronal block is
almost never cut perfectly perpendicular to the AP axis, so a single physical section
actually grazes a small *range* of AP levels across its width. ABBA estimates that tilt and
accounts for it. You will see the fingerprint of this in our pregnant section in a moment.

When ABBA is done, you export, among other things, a **per-pixel CCF coordinate map**: a
3-channel image where the three channels are the `x`, `y`, `z` atlas coordinates of every
pixel. That export is exactly the artifact our pipeline consumes.

### why we do not run ABBA in this notebook

We will not drive ABBA live, and you should understand why, because the reasoning is itself a
lesson in tool choice:

- ABBA is a **Windows-friendly desktop GUI**. Its core (elastix, BigWarp, DeepSlice) is
  Java and C++, so it **cannot be unrolled into transparent Python** the way every other step
  of this course can. It would be an opaque black box in a course built on reading the code.
- The course runs in Jupyter on Linux. The Python wrapper for ABBA is brittle and its headless
  mode is unreliable, so live-driving a mouse-heavy GUI for two sections is high effort, low
  teaching payoff.
- Registration is a **one-time setup**, not a method you need to master to do the science. The
  differential lipidomics, the MERFISH join, and the gene-to-lipid models are where the value is.

So in this course ABBA is **a concept with a described demo**: in your own lab you would
create a QuPath project, run DeepSlice then affine then spline elastix, fix slices with
BigWarp, and export the coordinate map. Here we hand you that output already.

The two paragraphs that follow describe how the *original* Lipid Brain Atlas was registered.
These are **published method facts, not numbers this notebook computes**, so treat them as
cited background and not as anything you can verify from the data we load. The full recipe is
in the atlas paper Methods (Fusar Bassini et al., *bioRxiv* 2025.10.13.682018, the registration
section). There, each section was placed onto the Allen CCFv3 with ABBA for the gross alignment
and then refined in-plane with **STalign**, a fully scriptable Python LDDMM tool (Clifton et al.,
*Nature Communications* 2023, doi:10.1038/s41467-023-43915-7). The STalign step aligned a lipid
image against an Allen cell-density image; the paper reports using **m/z 845.528**, the peak it
found **most correlated with cell density**, as that lipid channel. That STalign refinement is
the transparent, hands-on registration step; you will meet it in a later notebook. For now, we
load the result.

💡 **HINT.** If you ever do this yourself: the Allen CCFv3 and the atlas-region lookup are also
reachable programmatically through the `allensdk` and `brainglobe` Python packages, which is
how the per-pixel coordinates in our file were turned into region names and colors in the first
place. We keep those calls out of the live notebook because they download large atlas files on
first use; the result is baked into the data you load next.

## 4 · loading the per-pixel CCF coordinates

Time to look at the real output. We load the two-section pair. Everything ABBA and the Allen
lookup produced is already sitting in `adata.obs`: the CCF coordinates, the integer voxel
indices, and the Allen region name, acronym, and color for every pixel.

🔬 **TASK.** Load the substrate and confirm its shape: 174,768 pixels by 173 lipids.


In [ ]:
# the two-section pair: control 217D + pregnant Brain1_C2, same coronal plane (~AP 6.5)
adata = ad.read_h5ad("../../data/sections_pair.h5ad")
print("pixels x lipids:", adata.shape)
print("conditions:")
print(adata.obs["Condition"].value_counts())
print("\nsections (SectionID):")
print(adata.obs["SectionID"].value_counts())


The anatomy lives in a handful of `obs` columns. Let us look at exactly the ones this
notebook is about, for a few pixels.

🔬 **TASK.** Print the CCF coordinates, the integer voxel indices, and the Allen labels for
the first few pixels.


In [ ]:
# the anatomy columns produced by registration + Allen lookup
anat_cols = ["Condition",
             "xccf", "yccf", "zccf",        # CCF coordinate, in millimeters
             "x_index", "y_index", "z_index",  # integer voxel index into the annotation volume
             "acronym", "name", "allencolor"]  # the region label, full name, and color
adata.obs[anat_cols].head(6)


Read one row. `xccf, yccf, zccf` are the pixel's position in the Allen brain, in
millimeters along the three anatomical axes. `x_index, y_index, z_index` are the same position
expressed as **integer voxel indices** on the 25-micrometer grid. `acronym` is the region's
short code, `name` is its full label, `allencolor` is the Allen palette color as a hex string.

❓ **QUESTION.** There are three CCF numbers but each pixel only sits on a 2D section. Where do
all three come from? The two **in-plane** coordinates come from where the pixel landed after
the warp. The **out-of-plane** one (here `xccf`, the AP axis) comes from which atlas level the
section was placed at, plus the slicing-angle correction. Together they pin the pixel in 3D.


## 5 · how a coordinate indexes the Allen annotation volume

Here is the bridge from a coordinate to a region, made completely explicit. The CCF is a grid
of 25-micrometer voxels. A millimeter is 1000 micrometers, so one millimeter spans
**1000 / 25 = 40 voxels**. To turn a coordinate in millimeters into a voxel index, you multiply
by 40 and take the floor (drop the fractional part), because a coordinate anywhere inside a
voxel belongs to that voxel.

$$ \text{index} = \big\lfloor \text{ccf}_\text{mm} \times 40 \big\rfloor. $$

That integer triple `(x_index, y_index, z_index)` is then the address you look up in the Allen
annotation volume to read off the region. Let us verify that the indices in our file are
exactly this floor of `ccf * 40`. If they are, you understand precisely how the lookup worked.

🔬 **TASK.** Recompute the voxel indices from the CCF coordinates and check they match the
stored indices.


In [ ]:
# index = floor(ccf_mm * 40), because 1 mm = 40 voxels at 25 um
for ccf, idx in [("xccf", "x_index"), ("yccf", "y_index"), ("zccf", "z_index")]:
    recomputed = np.floor(adata.obs[ccf].to_numpy() * 40)
    match = recomputed == adata.obs[idx].to_numpy()
    frac_match = np.mean(match)
    n_mismatch = int(np.sum(~match))
    # print 4 decimals so the rare floating-point mismatches are visible, not rounded to 100%
    print(f"{ccf} * 40 (floored) matches {idx} for {frac_match*100:.4f}% of pixels "
          f"({n_mismatch} mismatch of {len(match)})")

⚠️ **CHECKPOINT.** The x and y axes match for 100.0000% of pixels, and the z axis matches for
99.9989%, which is exactly 2 pixels out of 174,768 that disagree by a single voxel. That tiny
discrepancy is normal floating-point dust: a coordinate sitting right on a voxel boundary can
round one way during the original lookup and the other way when we recompute it. So the chain
is exactly: **registration gives `(xccf, yccf, zccf)` in mm → multiply by 40, floor → integer
voxel index → look up that voxel in the Allen annotation volume → out comes `acronym`, `name`,
`allencolor`.** Nothing mysterious, just a coordinate indexing a labeled 3D array.

Now the slicing-angle point from section 3, made concrete. A perfectly coronal section sits at
one single AP level, so its `x_index` should take exactly one value. A section cut at a slight
tilt grazes a small range of AP levels across its width, so its `x_index` spreads over several
values. Let us check our two sections.

🔬 **TASK.** Count how many distinct AP levels (`x_index` values) each section spans, and the
millimeter range of its AP coordinate.

In [ ]:
# AP axis is x here: a flat coronal cut -> one x_index; a tilted cut -> a small range
for cond in ["naive", "pregnant"]:
    sub = adata.obs[adata.obs["Condition"] == cond]
    n_levels = sub["x_index"].nunique()
    lo, hi = sub["xccf"].min(), sub["xccf"].max()
    print(f"{cond:9s}: {n_levels:2d} distinct AP level(s), "
          f"xccf from {lo:.3f} to {hi:.3f} mm")


⚠️ **CHECKPOINT.** The control section sits at a single AP level (one `x_index`, a flat
coronal plane), while the pregnant section spreads across about three dozen AP levels over
roughly a millimeter. That spread is the **slicing-angle correction** doing its job: ABBA
recognized that the pregnant block was cut at a small tilt and assigned each part of the
section to the AP level it actually belongs to, rather than forcing the whole thing onto one
plane. Both sections still center on the same plane near AP 6.5, so they are genuinely
comparable; the tilt is just honestly accounted for.


## 6 · region maps: control vs pregnant

Now the payoff. Every pixel carries an Allen color in `allencolor`, so coloring pixels by it
draws the brain regions directly onto the section, in the standard Allen palette. We use the
helper `cl.plotting.spatial_categorical`, which scatters each pixel at its CCF position
`(zccf, -yccf)` and paints it with the stored hex color. (We plot `-yccf` so that dorsal is up,
matching how you would view a coronal section.)

Before calling the helper, look at what it does in one line: it is just a scatter where the
color of each point is read straight from a column of hex strings. No colormap, no
normalization, because the colors are already chosen by the Allen atlas.

🔬 **TASK.** Draw the Allen region map for both sections side by side.


In [ ]:
# region map: each pixel painted with its Allen region color (allencolor)
axes = plotting.spatial_categorical(adata, color_key="allencolor",
                                    section_key="SectionID", title_key="Condition",
                                    point_size=3.0)
axes[0].figure.suptitle("Allen regions on each section (control vs pregnant)",
                        y=1.02, fontsize=FS["l"])
plt.show()


⚠️ **CHECKPOINT.** Two coronal sections, each carved into colored territories: the big
central blob is the caudoputamen, the folded outer ribbon is cortex, the paired structures
below are hippocampus and thalamus. The two sections show the **same anatomy in the same colors**,
because both were registered into the same atlas. That shared frame is exactly what lets us
compare them.

❓ **QUESTION.** The two sections do not look pixel-for-pixel identical: tissue is torn here,
missing there, a little rotated. Yet the colored regions land in the same places. What made
that possible? The registration: each raw, distorted section was warped onto the common atlas,
so anatomy, not slide position, decides a pixel's color.

Let us confirm how many distinct Allen regions we are working with, and which are the largest.


In [ ]:
# how many Allen regions appear, and the most-sampled ones
print("distinct Allen regions on these two sections:", adata.obs["acronym"].nunique())
print("\nlargest regions by pixel count:")
print(adata.obs["acronym"].value_counts().head(10))


So our two sections together touch 174 distinct Allen regions. The caudoputamen (`CP`) is
the biggest, then piriform cortex (`PIR`), then a hippocampal field (`CA3`), exactly the
structures you expect at this coronal level. Each of these is now a group we can average lipids
over.


## 7 · the grey matter vs white matter split

The single biggest division in brain lipid composition is **grey matter vs white matter**. White
matter is dense with **myelin**, the lipid-rich insulating sheath that oligodendrocytes wrap
around axons, and myelin is built largely from sphingolipids like **HexCer** (hexosylceramides)
and **Cer** (ceramides). Grey matter, packed with cell bodies and synapses, is richer in
glycerophospholipids. If a lipidomic method cannot see the grey/white split, it cannot see
anything; it is the first sanity check.

The Allen atlas already encodes this split, but not in the `division` column of our file (here
that column is uninformative). It is encoded in the **structure path**: every region stores its
full ancestry in the Allen tree as a list of parent IDs, in the column `structure_id_path`. The
Allen tree has a top-level node for **fiber tracts** (white matter) with ID **1009**. So a pixel
is white matter exactly when **1009 appears in its structure path**. This is how you ask the
ontology a structured question instead of guessing from region names.

🔬 **TASK.** Build a grey/white label by checking whether the white-matter ID 1009 is in each
pixel's structure path, and count how the pixels split.


In [ ]:
# the structure path is stored as a string like "[997, 8, 567, ...]" -> parse it
WHITE_MATTER_ID = 1009  # Allen 'fiber tracts' top-level node

def is_white_matter(path_str):
    ids = ast.literal_eval(path_str)      # turn the string into a real list of ints
    return WHITE_MATTER_ID in ids         # white matter if 1009 is an ancestor

adata.obs["matter"] = np.where(
    adata.obs["structure_id_path"].apply(is_white_matter), "white", "grey")

print(adata.obs["matter"].value_counts())
print(f"\nwhite matter is {100 * (adata.obs['matter'] == 'white').mean():.1f}% of all pixels")


About a tenth of the pixels are white matter, the rest grey, which is the right ballpark
for a coronal section at this level (the corpus callosum, internal capsule, and other tracts
cut through here). Now draw the split on the sections so you can *see* the tracts.

🔬 **TASK.** Map a simple two-color scheme: black for white matter, light gray for grey
matter, on both sections.


In [ ]:
# paint white matter black, grey matter light-gray, to expose the tracts
matter_color = {"white": "#111111", "grey": "#cfcfcf"}
adata.obs["matter_color"] = adata.obs["matter"].map(matter_color)

axes = plotting.spatial_categorical(adata, color_key="matter_color",
                                    section_key="SectionID", title_key="Condition",
                                    point_size=3.0)
axes[0].figure.suptitle("grey (light) vs white matter (black)", y=1.02, fontsize=FS["l"])
plt.show()


⚠️ **CHECKPOINT.** The black pixels trace clear bands: the arc of the corpus callosum over
the top, the internal capsule cutting through the caudoputamen, the fimbria near the
hippocampus. These are the white-matter tracts, recovered purely from the Allen ontology, with
no lipid information used. Next we check whether the lipids agree.


### does myelin lipid track the white-matter anatomy?

Here is the test that ties anatomy to chemistry. If white matter is myelin and myelin is
sphingolipid, then a myelin marker lipid like **HexCer 42:2;O2** should be far brighter in the
white-matter pixels than in grey. That is the exact feature name as it appears in
`adata.var_names`: `HexCer` is the class (hexosylceramide), `42:2` is the sum composition
(42 acyl carbons, 2 double bonds), and the `;O2` suffix records two extra backbone oxygens.
From here we call it HexCer 42:2 for short, but it is the same molecule. We never told the
lipids where the tracts are; if they light up the tracts anyway, anatomy and lipidomics are
telling the same story.

🔬 **TASK.** Compare the mean intensity of HexCer 42:2 in white vs grey matter, then plot the
lipid spatially so you can see it overlap the tracts.

In [ ]:
# myelin marker: HexCer 42:2;O2. Compare its mean in white vs grey matter.
myelin_lipid = "HexCer 42:2;O2"
j = list(adata.var_names).index(myelin_lipid)
vals = np.asarray(adata.X[:, j]).ravel()

for matter in ["white", "grey"]:
    m = (adata.obs["matter"] == matter).to_numpy()
    print(f"{myelin_lipid} mean in {matter} matter: {vals[m].mean():.4f}")

ratio = vals[(adata.obs['matter'] == 'white').to_numpy()].mean() / \
        vals[(adata.obs['matter'] == 'grey').to_numpy()].mean()
print(f"\nwhite / grey ratio: {ratio:.2f}x")


In [ ]:
# now SEE it: the helper draws one lipid on both sections with a shared color scale
plotting.spatial_lipid(adata, myelin_lipid, section_key="SectionID",
                       title_key="Condition", point_size=3.0,
                       background=False, show_contours=False)
plt.show()


⚠️ **CHECKPOINT.** HexCer 42:2 is roughly three times brighter in white matter than in grey,
and the spatial map lights up exactly the bands you saw in black a moment ago: the corpus
callosum, the internal capsule, the fimbria. The myelin lipid traces the white-matter anatomy
on its own. That agreement between an anatomy label (from the Allen atlas) and a lipid signal
(from the mass spectrometer), with neither one informed by the other, is the first proof that
the registration is sound and that lipids carry real anatomical structure.


## 8 · region-level lipid views: the mean lipid profile of every region

We have a region label on every pixel and 173 lipid intensities per pixel. The natural summary
is the **region-by-lipid matrix**: for each Allen region, the mean intensity of each lipid
across all its pixels. One row per region, one column per lipid. This single table is the
workhorse for everything anatomical that follows: it is what a region-level differential test
compares between conditions, and it is the shape that matches the MERFISH region-by-gene table
we will join later.

Before any helper, build it by hand in two lines, because the operation is just a `groupby`
mean. Seeing it built makes the helper obvious.

🔬 **TASK.** Construct the region-by-lipid mean matrix with a plain pandas groupby.


In [ ]:
# region x lipid: mean intensity of each lipid within each Allen region
X = np.asarray(adata.X)
lipid_df = pd.DataFrame(X, columns=list(adata.var_names))
lipid_df["acronym"] = adata.obs["acronym"].to_numpy()

region_by_lipid = lipid_df.groupby("acronym", observed=True).mean()
print("region x lipid matrix shape:", region_by_lipid.shape)
region_by_lipid.iloc[:5, :4]   # a corner of the table: 5 regions x 4 lipids


That is the whole idea: 174 regions, 173 lipids, each entry a mean intensity. The helper
`cl.plotting.sorted_lipid_heatmap` does exactly this groupby mean and then draws it as a
heatmap, with one extra touch: it orders both the rows (regions) and the columns (lipids) by
hierarchical clustering on cosine similarity, so that regions with similar lipid profiles sit
next to each other and the block structure becomes visible. The sorting is the art; the table
underneath is the two-line groupby you just wrote.

🔬 **TASK.** Draw the region-by-lipid heatmap, clustered on both axes.


In [ ]:
# the helper: per-lipid 0-1 normalize, group-mean by acronym, then cosine-cluster both axes
ax, sorted_df = plotting.sorted_lipid_heatmap(
    adata, group_key="acronym", cmap="magma",
    figsize=(16, 9), title="region x lipid (mean intensity, both axes clustered)")
plt.show()
print("clustered matrix shape:", sorted_df.shape)


⚠️ **CHECKPOINT.** The heatmap shows clear **blocks**: groups of regions (rows) that share a
group of high lipids (columns). Those blocks are the lipidomic signature of anatomy. A band of
regions lighting up the sphingolipid columns is the white matter; other blocks are cortical,
striatal, thalamic. Regions that landed next to each other did so because their lipid profiles
are similar, which is the clustering finding structure with no anatomy labels used in the
sorting at all.

❓ **QUESTION.** Why average lipids within a region instead of testing pixel by pixel? Two
reasons. First, a single pixel is noisy and is a *mixture* of cell bodies, axons, and glia, not
a clean readout; averaging over a region cancels that noise. Second, the MERFISH dataset we
join later is summarized **per Allen region** (670 regions by 8460 genes), so to ask "do the
lipids and the genes agree in the same region?" we need the lipids in the same per-region
shape. Anatomy is the shared key that lets two completely different measurement technologies
speak to each other.


## 9 · why this anatomy powers the next notebooks

Step back and see what we built and why it matters downstream.

- We loaded **per-pixel CCF coordinates** that registration (ABBA, refined by STalign) produced,
  and saw exactly how a coordinate indexes the Allen annotation volume to give every pixel a
  region.
- We drew **region maps** for control and pregnant in the same atlas frame, so the two sections
  are directly comparable region by region.
- We split **grey from white matter** straight from the Allen ontology, and confirmed the myelin
  lipid HexCer 42:2 traces the white-matter tracts on its own, the first proof that anatomy and
  lipidomics agree.
- We built the **region-by-lipid matrix**, the mean lipid profile of every region.

That region-by-lipid matrix is the hinge for what comes next:

- **region-level differential lipidomics.** With both sections in the same atlas, we can ask, for
  each region, which lipids changed between control and pregnant, using the Wilcoxon +
  Benjamini-Hochberg test you have already met, region by region.
- **the MERFISH gene join.** The MERFISH dataset gives gene expression averaged over the same
  Allen regions (670 regions by 8460 genes). The Allen acronym is the shared key: join our
  region-by-lipid table to the region-by-gene table on `acronym`, and we can finally ask whether
  a region's lipid profile and its gene-expression profile tell the same story. Let us confirm
  the key actually lines up.

🔬 **TASK.** Load the MERFISH region-by-gene table and count how many of our Allen regions it
shares, so you see the join is real.


In [ ]:
# the MERFISH region x gene table, indexed by Allen acronym -> the future join key
merfish = pd.read_parquet("../../avemerfish_imputed_named.parquet")
print("MERFISH region x gene table:", merfish.shape, "(regions x genes)")

our_regions = set(adata.obs["acronym"].dropna().unique())
merfish_regions = set(merfish.index.astype(str))
shared = our_regions & merfish_regions
print(f"\nour regions: {len(our_regions)}")
print(f"MERFISH regions: {len(merfish_regions)}")
print(f"shared acronyms (the join key): {len(shared)}")
print("examples:", sorted(shared)[:8])


⚠️ **CHECKPOINT.** The MERFISH table is 670 regions by 8460 genes, and 156 of our 174
regions appear in it by acronym. That overlap is the bridge: every region where we have both a
mean lipid profile and a mean gene profile is a place we can directly compare lipids to genes,
all because both were registered into the same Allen atlas. Anatomy is not decoration; it is
the common language that makes the whole multi-omic story possible.

You now know what a reference atlas is, how registration (affine plus nonlinear warp, done by
ABBA in practice) puts every pixel on it, how a coordinate indexes the annotation volume, and
how to turn pixels into region-level views. The next notebook puts this to work on the biology:
which lipids change, region by region, in pregnancy.


## References

- Fusar Bassini et al. *The lipidomic architecture of the mouse brain.* bioRxiv 2025.10.13.682018. The EUCLID method and the Lipid Brain Atlas: https://www.biorxiv.org/content/10.1101/2025.10.13.682018v1
- uMAIA: probabilistic normalization of mass-spectrometry imaging. *Nature Methods* (2025), s41592-025-02771-7.
- Allen Mouse Brain Common Coordinate Framework v3 (CCFv3), Allen Institute for Brain Science.
- Explore the atlas interactively: https://lbae-v2.epfl.ch/
